# 🏠 House Price Prediction
> Proyek ini bertujuan memprediksi harga rumah menggunakan dataset yang berisi 13 fitur utama seperti lokasi, ukuran, dan kondisi bangunan. Tiga algoritma machine learning dibandingkan: **SVR**, **Random Forest**, dan **Linear Regression**.

---

## 📦 1. Import Library

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, r2_score
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

# Pengaturan tampilan
pd.set_option('display.max_columns', None)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid', palette='muted')

print("✅ Semua library berhasil diimpor.")

## 📂 2. Memuat Dataset
Dataset berisi informasi harga jual rumah beserta fitur-fitur properti. Berikut deskripsi kolom:

| Kolom | Deskripsi |
|---|---|
| `Id` | Identifikasi unik tiap record |
| `MSSubClass` | Tipe bangunan dalam transaksi |
| `MSZoning` | Klasifikasi zona properti |
| `LotArea` | Luas lahan (sqft) |
| `LotConfig` | Konfigurasi lahan |
| `BldgType` | Tipe bangunan |
| `OverallCond` | Rating kondisi keseluruhan |
| `YearBuilt` | Tahun konstruksi |
| `YearRemodAdd` | Tahun renovasi terakhir |
| `Exterior1st` | Material eksterior |
| `BsmtFinSF2` | Luas basement tipe 2 (sqft) |
| `TotalBsmtSF` | Total luas basement (sqft) |
| `SalePrice` | ⭐ Target: harga jual rumah |

In [ ]:
dataset = pd.read_excel('HousePricePrediction.xlsx')

print(f"📐 Dimensi dataset : {dataset.shape[0]:,} baris × {dataset.shape[1]} kolom")
print(f"💾 Ukuran memori   : {dataset.memory_usage(deep=True).sum() / 1024:.1f} KB\n")
dataset.head(10)

## 🔍 3. Exploratory Data Analysis (EDA)

### 3.1 Informasi Tipe Data

In [ ]:
# Ringkasan tipe data
object_cols  = dataset.select_dtypes(include=['object', 'str']).columns.tolist()
int_cols     = dataset.select_dtypes(include=['int64']).columns.tolist()
float_cols   = dataset.select_dtypes(include=['float64']).columns.tolist()

print(f"🔤 Fitur Kategorikal : {len(object_cols)} → {object_cols}")
print(f"🔢 Fitur Integer     : {len(int_cols)}  → {int_cols}")
print(f"🔢 Fitur Float       : {len(float_cols)}  → {float_cols}")
print()
dataset.info()

### 3.2 Statistik Deskriptif

In [ ]:
dataset.describe(include='all').T

### 3.3 Missing Values

In [ ]:
missing = dataset.isnull().sum()
missing = missing[missing > 0].reset_index()
missing.columns = ['Kolom', 'Jumlah Missing']
missing['Persentase (%)'] = (missing['Jumlah Missing'] / len(dataset) * 100).round(2)

print("📋 Kolom dengan nilai kosong:")
print(missing.to_string(index=False))

# Visualisasi
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=missing, x='Kolom', y='Persentase (%)', palette='Reds_r', ax=ax)
    ax.set_title('Persentase Missing Values per Kolom', fontsize=13, fontweight='bold')
    ax.set_ylabel('Missing (%)')
    for bar, pct in zip(ax.patches, missing['Persentase (%)']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{pct}%', ha='center', fontsize=10)
    plt.tight_layout()
    plt.savefig('missing_values.png', bbox_inches='tight')
    plt.show()

### 3.4 Distribusi Target: SalePrice

In [ ]:
train_data = dataset.dropna(subset=['SalePrice'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
sns.histplot(train_data['SalePrice'], bins=50, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Distribusi SalePrice', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Harga Jual (USD)')
axes[0].axvline(train_data['SalePrice'].mean(), color='red', linestyle='--', label=f"Mean: ${train_data['SalePrice'].mean():,.0f}")
axes[0].legend()

# Boxplot
sns.boxplot(y=train_data['SalePrice'], color='lightcoral', ax=axes[1])
axes[1].set_title('Boxplot SalePrice', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Harga Jual (USD)')

plt.suptitle('Analisis Distribusi SalePrice', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('saleprice_distribution.png', bbox_inches='tight')
plt.show()

print(f"📊 Statistik SalePrice:")
print(f"   Mean   : ${train_data['SalePrice'].mean():>12,.0f}")
print(f"   Median : ${train_data['SalePrice'].median():>12,.0f}")
print(f"   Std    : ${train_data['SalePrice'].std():>12,.0f}")
print(f"   Min    : ${train_data['SalePrice'].min():>12,.0f}")
print(f"   Max    : ${train_data['SalePrice'].max():>12,.0f}")

### 3.5 Correlation Matrix (Fitur Numerik)

In [ ]:
numerical_dataset = train_data.select_dtypes(include=['int64', 'float64'])

plt.figure(figsize=(12, 8))
corr = numerical_dataset.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # tampilkan setengah saja

sns.heatmap(corr,
            mask=mask,
            annot=True,
            cmap='coolwarm',
            fmt='.2f',
            linewidths=0.5,
            linecolor='white',
            vmin=-1, vmax=1,
            square=True)
plt.title('Correlation Matrix (Fitur Numerik)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', bbox_inches='tight')
plt.show()

# Top korelasi dengan SalePrice
top_corr = corr['SalePrice'].drop('SalePrice').sort_values(ascending=False)
print("\n📈 Korelasi dengan SalePrice:")
print(top_corr.to_string())

### 3.6 Distribusi Fitur Kategorikal

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(object_cols):
    counts = dataset[col].value_counts()
    sns.barplot(x=counts.index, y=counts.values, palette='viridis', ax=axes[i])
    axes[i].set_title(f'Distribusi: {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Jumlah')
    axes[i].tick_params(axis='x', rotation=45)
    # Tambahkan label nilai
    for bar in axes[i].patches:
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     int(bar.get_height()), ha='center', va='bottom', fontsize=8)

plt.suptitle('Distribusi Fitur Kategorikal', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('categorical_distribution.png', bbox_inches='tight')
plt.show()

### 3.7 Hubungan Fitur Numerik dengan SalePrice

In [ ]:
num_features = [c for c in int_cols if c not in ['Id', 'SalePrice']]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].scatter(train_data[col], train_data['SalePrice'],
                    alpha=0.4, color='steelblue', edgecolors='none', s=15)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('SalePrice')
    axes[i].set_title(f'{col} vs SalePrice', fontsize=10, fontweight='bold')

# Sembunyikan axes yang tidak terpakai
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Scatter Plot: Fitur Numerik vs SalePrice', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('scatter_plots.png', bbox_inches='tight')
plt.show()

## ⚙️ 4. Data Preprocessing

### 4.1 Hapus Kolom ID & Tangani Missing Values

In [ ]:
# Drop kolom Id (tidak informatif untuk prediksi)
df = dataset.drop(columns=['Id'])

# Hanya gunakan baris yang memiliki SalePrice (data training)
df_train = df[df['SalePrice'].notna()].copy()

# Isi missing value numerik dengan median (lebih robust terhadap outlier)
num_cols = df_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols = [c for c in num_cols if c != 'SalePrice']
for col in num_cols:
    if df_train[col].isnull().any():
        df_train[col] = df_train[col].fillna(df_train[col].median())

# Isi missing value kategorikal dengan modus
cat_cols = df_train.select_dtypes(include=['object', 'str']).columns.tolist()
for col in cat_cols:
    if df_train[col].isnull().any():
        df_train[col] = df_train[col].fillna(df_train[col].mode()[0])

print(f"✅ Dataset training setelah preprocessing: {df_train.shape}")
print(f"\n🔍 Sisa missing values: {df_train.isnull().sum().sum()} (harus 0)")

### 4.2 Deteksi & Penanganan Outlier (SalePrice)

In [ ]:
Q1 = df_train['SalePrice'].quantile(0.25)
Q3 = df_train['SalePrice'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

outliers = df_train[(df_train['SalePrice'] < lower) | (df_train['SalePrice'] > upper)]
print(f"📌 Batas IQR  : ${lower:,.0f} – ${upper:,.0f}")
print(f"⚠️  Jumlah outlier SalePrice: {len(outliers)} baris ({len(outliers)/len(df_train)*100:.1f}%)")

# Tetap pertahankan outlier agar informasi tidak hilang
print("ℹ️  Outlier dipertahankan untuk menjaga informasi harga premium.")

### 4.3 Encoding Fitur Kategorikal (OneHotEncoding)

In [ ]:
oh_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')

oh_array = oh_encoder.fit_transform(df_train[cat_cols])
oh_df    = pd.DataFrame(oh_array,
                         columns=oh_encoder.get_feature_names_out(cat_cols),
                         index=df_train.index)

df_final = pd.concat([df_train.drop(columns=cat_cols), oh_df], axis=1)

print(f"✅ Shape setelah encoding: {df_final.shape}")
print(f"   Fitur baru dari OHE   : {oh_df.shape[1]}")
df_final.head(3)

## ✂️ 5. Pembagian Dataset (Train / Validation)

In [ ]:
X = df_final.drop(columns=['SalePrice'])
y = df_final['SalePrice']

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"📊 Total sampel  : {len(X):,}")
print(f"   Training      : {len(X_train):,} ({len(X_train)/len(X)*100:.0f}%)")
print(f"   Validasi      : {len(X_valid):,} ({len(X_valid)/len(X)*100:.0f}%)")
print(f"   Jumlah fitur  : {X.shape[1]}")

## 🤖 6. Pelatihan & Evaluasi Model

### 6.1 Helper: Fungsi Evaluasi

In [ ]:
def evaluate_model(name, y_true, y_pred):
    """Menghitung dan menampilkan metrik evaluasi regresi."""
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"{'─'*40}")
    print(f"  Model : {name}")
    print(f"  MAPE  : {mape:.2f}%")
    print(f"  MAE   : ${mae:,.0f}")
    print(f"  R²    : {r2:.4f}")
    print(f"{'─'*40}")
    return {'Model': name, 'MAPE (%)': round(mape, 2), 'MAE ($)': round(mae, 0), 'R²': round(r2, 4)}

results = []

### 6.2 Support Vector Regressor (SVR)

In [ ]:
model_svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)
model_svr.fit(X_train, y_train)
y_pred_svr = model_svr.predict(X_valid)

results.append(evaluate_model('SVR', y_valid, y_pred_svr))

### 6.3 Random Forest Regressor

In [ ]:
model_rfr = RandomForestRegressor(n_estimators=200, max_depth=None,
                                   min_samples_split=5, random_state=42, n_jobs=-1)
model_rfr.fit(X_train, y_train)
y_pred_rfr = model_rfr.predict(X_valid)

results.append(evaluate_model('Random Forest', y_valid, y_pred_rfr))

### 6.4 Linear Regression

In [ ]:
model_lr = LinearRegression()
model_lr.fit(X_train, y_train)
y_pred_lr = model_lr.predict(X_valid)

results.append(evaluate_model('Linear Regression', y_valid, y_pred_lr))

## 📊 7. Perbandingan Model

In [ ]:
results_df = pd.DataFrame(results).set_index('Model')
print("\n📋 Ringkasan Performa Model:")
print(results_df.to_string())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ['MAPE (%)', 'MAE ($)', 'R²']
colors  = ['#4C72B0', '#DD8452', '#55A868']

for ax, metric, color in zip(axes, metrics, colors):
    bars = ax.bar(results_df.index, results_df[metric], color=color, width=0.5, edgecolor='white')
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylabel(metric)
    ax.set_ylim(0, results_df[metric].max() * 1.2)
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + results_df[metric].max()*0.02,
                f'{val:.2f}', ha='center', fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Perbandingan Performa Model', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()

best = results_df['R²'].idxmax()
print(f"\n🏆 Model terbaik berdasarkan R²: {best} ({results_df.loc[best,'R²']:.4f})")

### 7.1 Visualisasi Prediksi vs Aktual

In [ ]:
preds = {
    'SVR'              : y_pred_svr,
    'Random Forest'    : y_pred_rfr,
    'Linear Regression': y_pred_lr,
}
colors = ['#4C72B0', '#DD8452', '#55A868']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, y_pred), color in zip(axes, preds.items(), colors):
    ax.scatter(y_valid, y_pred, alpha=0.4, s=15, color=color, edgecolors='none')
    lims = [min(y_valid.min(), y_pred.min()), max(y_valid.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Ideal (y=x)')
    ax.set_xlabel('Harga Aktual (USD)')
    ax.set_ylabel('Harga Prediksi (USD)')
    ax.set_title(f'{name}\nR² = {r2_score(y_valid, y_pred):.4f}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Prediksi vs Aktual – Semua Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('pred_vs_actual.png', bbox_inches='tight')
plt.show()

### 7.2 Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(model_rfr.feature_importances_, index=X.columns)
top15 = importances.nlargest(15).sort_values()

plt.figure(figsize=(10, 6))
top15.plot(kind='barh', color='#4C72B0', edgecolor='white')
plt.title('Top 15 Feature Importance (Random Forest)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()

## ✅ 8. Kesimpulan

### Ringkasan Proyek
Proyek ini berhasil membangun pipeline prediksi harga rumah end-to-end, mulai dari eksplorasi data hingga evaluasi model.

### Temuan Utama
- **Missing Values**: Kolom `SalePrice` (data test), `MSZoning`, `Exterior1st`, `BsmtFinSF2`, `TotalBsmtSF` memiliki nilai kosong yang ditangani dengan median/modus.
- **Korelasi**: `TotalBsmtSF` dan `YearBuilt` menunjukkan korelasi positif tertinggi terhadap `SalePrice`.
- **Model Terbaik**: Random Forest secara konsisten menghasilkan MAPE dan R² terbaik karena kemampuannya menangkap hubungan non-linear.

### Rekomendasi Pengembangan
| Aspek | Rekomendasi |
|---|---|
| Feature Engineering | Tambahkan fitur `AgeOfHouse = YearSold - YearBuilt` |
| Hyperparameter Tuning | Gunakan `GridSearchCV` atau `Optuna` untuk Random Forest |
| Model Lanjutan | Coba `XGBoost` / `LightGBM` untuk performa lebih baik |
| Cross-Validation | Terapkan K-Fold CV untuk estimasi performa yang lebih stabil |